# DC 親子関係 最適化モデル

仕入れ先への発注最小ロット $Q$ が大きい場合、全 DC が直接仕入れると平均在庫が $Q/2$ になり在庫コストが膨らむ。
DC 間で **親子関係** を設定すると、子 DC は週 1 回・需要量分だけ補充でき平均在庫が $d/2$ まで減る。

このモデルは **在庫削減メリット** と **DC 間輸送コスト増加** を天秤にかけ、週次コスト合計が最小になる親子関係を求める。

```
仕入れ先 ─[タリフA]─> 親DC ─[タリフB]─> 子DC
仕入れ先 ─[タリフA]─> 孤立DC
```

## 1. DC の役割

| 役割 | 仕入れ元 | ロット | 子を持つか |
|------|---------|--------|------------|
| 孤立 DC | 仕入れ先（直接） | Q（大） | なし |
| 親 DC   | 仕入れ先（直接） | Q（大） | あり |
| 子 DC   | 親 DC（経由）   | 週間需要量（小） | なし |

各 DC の役割は最適化によって決定される（事前に固定しない）。

## 2. 数理モデル

### パラメータ

| 記号 | 意味 | 単位 |
|------|------|------|
| $d_j$ | DC $j$ の週間需要量 | ケース/週 |
| $h_j$ | DC $j$ の在庫保管単価 | 円/ケース/日 |
| $Q_j$ | DC $j$ のロットサイズ | ケース |
| $a_j$ | 仕入れ先 → DC $j$ のタリフ A | 円/ケース |
| $b_{jk}$ | DC $j$ → DC $k$ のタリフ B（非対称） | 円/ケース |

### 決定変数

$$x_{jk} \in \{0, 1\} \quad (j \neq k)$$

$x_{jk} = 1$ のとき、DC $k$ を DC $j$ の子として割り当てる。

### 目的関数（週次コスト最小化）

**孤立 DC / 親 DC**（仕入れ先から直接補充）:

$$C_j^{\text{direct}} = 7 h_j \frac{Q_j}{2} + a_j \left(d_j + \sum_k x_{jk} d_k\right) + \sum_k x_{jk} b_{jk} d_k$$

**子 DC**（親 DC から補充）:

$$C_j^{\text{child}} = 7 h_j \frac{d_j}{2}$$

**全体の目的関数**:

$$\min \sum_j \left[ \left(1 - \sum_l x_{lj}\right) C_j^{\text{direct}} + \left(\sum_l x_{lj}\right) C_j^{\text{child}} \right]$$

### 制約条件

| # | 内容 | 式 |
|---|------|-----|
| C1 | 親は 1 つまで | $\sum_j x_{jk} \leq 1 \quad \forall k$ |
| C2 | 2 層固定（親 DC は子 DC になれない） | $x_{lj} + x_{jk} \leq 1 \quad \forall l, j, k$ |
| C3 | 自己割り当て禁止 | $x_{jj} = 0 \quad \forall j$ |

## 3. 入出力仕様

### 入力 Excel ファイル構成

**シート「DCマスタ」**

| 列 | 型 | 説明 |
|----|----|------|
| `dc_id` | str | DC 識別子 |
| `demand` | float | 週間需要量（ケース/週） |
| `holding_cost` | float | 在庫保管単価（円/ケース/日） |
| `lot_size` | float | ロットサイズ（ケース） |
| `tariff_from_supplier` | float | タリフ A（円/ケース） |

**シート「DC間タリフ」**

| 列 | 型 | 説明 |
|----|----|------|
| `from_dc_id` | str | 送り元 DC |
| `to_dc_id` | str | 送り先 DC |
| `tariff` | float | タリフ B（円/ケース） |

### 出力

```python
{
  "status": "Optimal" | "Infeasible" | ...,
  "total_weekly_cost": float,        # 週次コスト合計（円/週）
  "parent_of": {
    "DC-X": None,       # 孤立 DC または 親 DC
    "DC-Y": "DC-X",    # 子 DC（DC-X が親）
  }
}
```

---
## 4. セットアップ

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
from src.excel_loader import load_excel
from src.optimizer import solve

---
## 5. Excel からデータを読み込む

`data/sample.xlsx` にはシナリオ 1 のサンプルデータが入っています。

In [ ]:
dcs, dc_tariffs = load_excel("data/sample.xlsx")

print("=== DCマスタ ===")
display(pd.DataFrame(dcs))

In [ ]:
print("=== DC間タリフ ===")
display(pd.DataFrame(dc_tariffs))

---
## 6. 最適化を実行する

In [ ]:
result = solve(dcs, dc_tariffs)
result

---
## 7. 結果の確認

In [ ]:
def display_result(result):
    print(f"ステータス  : {result['status']}")
    print(f"週次コスト  : {result['total_weekly_cost']:,.0f} 円/週")
    print()
    print("親子関係:")
    for dc_id, parent in result["parent_of"].items():
        role = "孤立 DC / 親 DC" if parent is None else f"子 DC（親: {parent}）"
        print(f"  {dc_id}: {role}")

display_result(result)

---
## 8. シナリオ検証（SPEC.md 準拠）

SPEC.md に記載の 3 シナリオで期待値と一致するかを確認する。

In [ ]:
def run_scenario(title, dcs_data, tariffs_data, expected_cost, expected_parent_of):
    print(f"{'=' * 55}")
    print(f"  {title}")
    print(f"{'=' * 55}")
    res = solve(dcs_data, tariffs_data)
    display_result(res)

    cost_ok   = abs(res["total_weekly_cost"] - expected_cost) < 0.01
    parent_ok = res["parent_of"] == expected_parent_of
    verdict   = "PASS" if cost_ok and parent_ok else "FAIL"
    print(f"\n期待コスト: {expected_cost:,.0f} 円/週  [{verdict}]\n")

### シナリオ 1: 親子関係を作る方が最適（2 DC）

DC-B のタリフ A が高いため、DC-A を親にして DC-B を子にする構成が最適になる。
在庫削減効果（31,500 → 3,500 円/週）が輸送コスト増加（+400 円/週）を大きく上回る。

| パターン | DC-A | DC-B | 合計 |
|---------|------|------|------|
| 両方孤立 | 17,750 | 18,250 | **36,000** |
| DC-B が DC-A の子 | 18,000 | 1,900 | **19,900** ← 最適 |
| DC-A が DC-B の子 | 1,900 | 19,000 | **20,900** |

In [ ]:
run_scenario(
    "シナリオ 1: 親子関係を作る方が最適（2 DC）",
    dcs_data=[
        {"dc_id": "DC-A", "demand": 50, "holding_cost": 10, "lot_size": 500, "tariff_from_supplier": 5},
        {"dc_id": "DC-B", "demand": 50, "holding_cost": 10, "lot_size": 500, "tariff_from_supplier": 15},
    ],
    tariffs_data=[
        {"from_dc_id": "DC-A", "to_dc_id": "DC-B", "tariff": 3},
        {"from_dc_id": "DC-B", "to_dc_id": "DC-A", "tariff": 3},
    ],
    expected_cost=19900.0,
    expected_parent_of={"DC-A": None, "DC-B": "DC-A"},
)

### シナリオ 2: 全 DC 孤立が最適（2 DC）

DC 間タリフ B が非常に高い（70 円/ケース）ため、
在庫削減効果（3,150 円/週）を輸送コスト増加（3,500 円/週）が上回り、全孤立が最適になる。

| パターン | DC-A | DC-B | 合計 |
|---------|------|------|------|
| 両方孤立 | 3,750 | 3,750 | **7,500** ← 最適 |
| DC-B が DC-A の子 | 4,000 | 3,850 | **7,850** |
| DC-A が DC-B の子 | 3,850 | 4,000 | **7,850** |

In [ ]:
run_scenario(
    "シナリオ 2: 全 DC 孤立が最適（2 DC）",
    dcs_data=[
        {"dc_id": "DC-A", "demand": 50, "holding_cost": 2, "lot_size": 500, "tariff_from_supplier": 5},
        {"dc_id": "DC-B", "demand": 50, "holding_cost": 2, "lot_size": 500, "tariff_from_supplier": 5},
    ],
    tariffs_data=[
        {"from_dc_id": "DC-A", "to_dc_id": "DC-B", "tariff": 70},
        {"from_dc_id": "DC-B", "to_dc_id": "DC-A", "tariff": 70},
    ],
    expected_cost=7500.0,
    expected_parent_of={"DC-A": None, "DC-B": None},
)

### シナリオ 3: DC-C を親にした集約が最適（3 DC）

DC-C は保管単価 h=2 が低く、大ロット在庫コストが小さい。
DC-C ↔ 他の輸送タリフは高い（70 円/ケース）が、DC-A・DC-B の在庫削減効果（各 14,000 円/週）がそれを上回る。

| パターン | 合計 |
|---------|------|
| 全孤立 | **41,000** |
| DC-B が DC-A の子・DC-C 孤立 | **26,300** |
| DC-A・DC-B が DC-C の子 | **26,000** ← 最適 |

In [ ]:
run_scenario(
    "シナリオ 3: DC-C を親にした集約が最適（3 DC）",
    dcs_data=[
        {"dc_id": "DC-A", "demand": 100, "holding_cost": 10, "lot_size": 500, "tariff_from_supplier": 5},
        {"dc_id": "DC-B", "demand": 100, "holding_cost": 10, "lot_size": 500, "tariff_from_supplier": 15},
        {"dc_id": "DC-C", "demand": 100, "holding_cost": 2,  "lot_size": 500, "tariff_from_supplier": 5},
    ],
    tariffs_data=[
        {"from_dc_id": "DC-A", "to_dc_id": "DC-B", "tariff": 3},
        {"from_dc_id": "DC-B", "to_dc_id": "DC-A", "tariff": 3},
        {"from_dc_id": "DC-A", "to_dc_id": "DC-C", "tariff": 70},
        {"from_dc_id": "DC-C", "to_dc_id": "DC-A", "tariff": 70},
        {"from_dc_id": "DC-B", "to_dc_id": "DC-C", "tariff": 70},
        {"from_dc_id": "DC-C", "to_dc_id": "DC-B", "tariff": 70},
    ],
    expected_cost=26000.0,
    expected_parent_of={"DC-A": "DC-C", "DC-B": "DC-C", "DC-C": None},
)